# 02 — ML Experiments
All assets in parallel. Uses WaveletFeatureExtractor (db2, 3 levels) to build a scalar feature vector per sliding window. joblib.Parallel dispatches (ticker × model) combinations.

In [ ]:
import sys, warnings, json, time
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../..").resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
import pywt
from joblib import Parallel, delayed
from scipy import stats
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from tqdm.notebook import tqdm

from config.experiment_config import (
    TICKERS, ML_MODELS_CONFIG, ML_N_JOBS_OUTER, VALIDATION_CONFIG, BACKTEST_CONFIG
)
from src.data_loader import load_processed, load_labels
from src.evaluation import ClassificationEvaluator, FinancialMetrics, ResultsManager
from src.backtest import simulate_strategy, buy_and_hold_returns

RESULTS_DIR = Path("results")

## 1. Wavelet Feature Extractor

In [ ]:
def extract_wavelet_features(window: np.ndarray, wavelet: str = "db2", level: int = 3) -> np.ndarray:
    """
    For each feature channel in window (seq_len, n_features),
    decompose with DWT and extract 17 stats per decomposition level.
    Returns a flat feature vector.
    """
    stats_fns = [
        np.mean, np.std, np.var, np.max, np.min,
        lambda x: np.max(x) - np.min(x),   # range
        np.median,
        lambda x: np.sum(x**2),             # energy
        lambda x: -np.sum((x**2 + 1e-10) * np.log(x**2 + 1e-10)),  # entropy
        lambda x: float(stats.skew(x)),
        lambda x: float(stats.kurtosis(x)),
        lambda x: float(np.percentile(x, 75) - np.percentile(x, 25)),  # IQR
        lambda x: float(np.median(np.abs(x - np.median(x)))),           # MAD
        lambda x: float(np.sqrt(np.mean(x**2))),                        # RMS
        lambda x: float(np.max(x) - np.min(x)),                         # peak-to-peak
        lambda x: float(np.sum(np.diff(np.sign(x)) != 0)),              # zero crossings
        lambda x: float(np.sum(np.diff(np.sign(x - np.mean(x))) != 0)), # mean crossings
    ]
    all_feats = []
    for ch in range(window.shape[1]):
        signal = window[:, ch].astype(float)
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        for coef in coeffs:
            for fn in stats_fns:
                try:
                    val = float(fn(coef))
                    all_feats.append(val if np.isfinite(val) else 0.0)
                except Exception:
                    all_feats.append(0.0)
    return np.array(all_feats, dtype=np.float32)


def build_ml_features(X_windows: np.ndarray, wavelet: str = "db2", level: int = 3) -> np.ndarray:
    """Apply extract_wavelet_features to all windows. Returns (N, D)."""
    return np.stack([extract_wavelet_features(w, wavelet, level) for w in X_windows])


print(f"Feature extractor ready. n_jobs_outer={ML_N_JOBS_OUTER}")

## 2. ML Model Factory

In [ ]:
def build_ml_model(model_name: str):
    cfg = ML_MODELS_CONFIG.get(model_name, {})
    if model_name == "RandomForest":
        return RandomForestClassifier(**{k: v for k, v in cfg.items()})
    if model_name == "XGBoost":
        params = {k: v for k, v in cfg.items() if k not in ("use_label_encoder", "eval_metric")}
        return XGBClassifier(**params, verbosity=0)
    if model_name == "LightGBM":
        return LGBMClassifier(**{k: v for k, v in cfg.items()})
    if model_name == "CatBoost":
        return CatBoostClassifier(**{k: v for k, v in cfg.items()})
    if model_name == "Stacking":
        estimators = [
            ("rf",  RandomForestClassifier(n_estimators=100, n_jobs=4, random_state=42)),
            ("xgb", XGBClassifier(n_estimators=100, nthread=4, verbosity=0, random_state=42)),
            ("lgb", LGBMClassifier(n_estimators=100, num_threads=4, verbose=-1, random_state=42)),
        ]
        return StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(max_iter=1000, multi_class="multinomial"),
            n_jobs=4,
            passthrough=False,
        )
    raise ValueError(f"Unknown model: {model_name}")

## 3. Single-Job Runner (ticker × model)

In [ ]:
def run_ml_job(ticker: str, model_name: str) -> dict:
    """Run one (ticker, model) job with PurgedKFold. Returns metrics dict."""
    from utils.validation import PurgedKFold

    result_path = RESULTS_DIR / ticker / f"{model_name}_ml" / "metrics.json"
    if result_path.exists():
        return {"ticker": ticker, "model_name": model_name, "status": "skipped"}

    try:
        features = load_processed(ticker)
        labels   = load_labels(ticker)
        common   = features.index.intersection(labels.index)
        features = features.loc[common].dropna()
        labels   = labels.loc[features.index]

        seq_len = VALIDATION_CONFIG.get("sequence_length", 30)
        n = len(features)
        idx = np.arange(seq_len, n)
        X_win = np.stack([features.values[i - seq_len: i] for i in idx])
        y_win = labels.values[idx]
        p_win = features["log_return_1"].values[idx] if "log_return_1" in features.columns else np.zeros(len(idx))

        # Wavelet feature extraction
        X_feat = build_ml_features(X_win)

        # PurgedKFold
        pct_embargo = VALIDATION_CONFIG.get("embargo_days", 10) / len(y_win)
        pkf = PurgedKFold(n_splits=VALIDATION_CONFIG.get("n_folds", 5), pct_embargo=pct_embargo)

        ml_folds, fin_folds = [], []
        y_true_all, y_pred_all = [], []

        for train_val_idx, test_idx in pkf.split(X_feat, y_win):
            n_val = max(1, int(len(train_val_idx) * 0.15))
            train_idx = train_val_idx[:-n_val]

            scaler = RobustScaler()
            X_tr = scaler.fit_transform(X_feat[train_idx])
            X_te = scaler.transform(X_feat[test_idx])
            y_tr = y_win[train_idx]
            y_te = y_win[test_idx]

            model = build_ml_model(model_name)
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)

            ml_folds.append(ClassificationEvaluator.evaluate(y_te, y_pred))
            strat = simulate_strategy(y_pred, pd.Series(p_win[test_idx]),
                                      transaction_cost=BACKTEST_CONFIG["transaction_cost"],
                                      allow_short=BACKTEST_CONFIG["allow_short"])
            bh = buy_and_hold_returns(pd.Series(p_win[test_idx]))
            fin_folds.append(FinancialMetrics.compute(strat, bh))
            y_true_all.append(y_te)
            y_pred_all.append(y_pred)

        # Aggregate
        agg_ml  = {k: float(np.mean([f[k] for f in ml_folds])) for k in ml_folds[0]}
        agg_fin = FinancialMetrics.aggregate_cv(fin_folds)

        # Save
        rm = ResultsManager(RESULTS_DIR)
        rm.log_experiment(ticker, model_name, "ml", agg_ml, agg_fin,
                          config={"wavelet": "db2", "level": 3})
        np.savez(result_path.parent / "predictions.npz",
                 y_true=np.concatenate(y_true_all),
                 y_pred=np.concatenate(y_pred_all))

        return {"ticker": ticker, "model_name": model_name, "status": "done",
                "f1_macro": agg_ml.get("f1_macro", float("nan")),
                "sharpe": agg_fin.get("fin_sharpe", float("nan"))}

    except Exception as exc:
        import traceback
        return {"ticker": ticker, "model_name": model_name, "status": "error",
                "error": traceback.format_exc()[-300:]}

## 4. Run All Jobs in Parallel

In [ ]:
ML_MODELS_TO_RUN = ["RandomForest", "XGBoost", "LightGBM", "CatBoost", "Stacking"]

combos = [(ticker, model) for ticker in TICKERS for model in ML_MODELS_TO_RUN]
print(f"Total jobs: {len(combos)}  |  n_jobs_outer: {ML_N_JOBS_OUTER}")

t0 = time.time()
results = Parallel(n_jobs=ML_N_JOBS_OUTER, backend="loky", verbose=5)(
    delayed(run_ml_job)(ticker, model) for ticker, model in combos
)
elapsed = time.time() - t0
print(f"\nCompleted {len(results)} jobs in {elapsed:.1f}s")

## 5. Results Summary

In [ ]:
results_df = pd.DataFrame(results)
print("Status counts:")
print(results_df["status"].value_counts())

errors = results_df[results_df["status"] == "error"]
if len(errors):
    print(f"\n{len(errors)} errors:")
    for _, row in errors.iterrows():
        print(f"  {row['ticker']} / {row['model_name']}: {row.get('error','')[:200]}")

done = results_df[results_df["status"] == "done"].copy()
if len(done):
    print("\nTop results by F1-macro:")
    print(done.nlargest(10, "f1_macro")[["ticker","model_name","f1_macro","sharpe"]].to_string())

In [ ]:
rm = ResultsManager(RESULTS_DIR)
all_results = rm.load_all_results()
ml_results = all_results[all_results["mode"] == "ml"]

if len(ml_results):
    pivot = ml_results.pivot_table(
        index="ticker", columns="model_name",
        values="ml_f1_macro", aggfunc="mean"
    ).round(3)
    print("F1-macro by ticker × model:")
    print(pivot.to_string())